<a href="https://colab.research.google.com/github/AgnelFernando/HealthAIAgent/blob/master/notebooks/Embeddings%2BVector_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install sentence-transformers psycopg2-binary tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 55.3 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
import psycopg2
from psycopg2.extras import RealDictCursor
from google.colab import userdata


DB_URL = userdata.get('DATABASE_URL')

conn = psycopg2.connect(DB_URL)
cur = conn.cursor(cursor_factory=RealDictCursor)

cur.execute("""
    select id, chunk_text
    from knowledge_chunks
    where embedding is null
""")

rows = cur.fetchall()
conn.close()

print("Chunks to embed:", len(rows))

Chunks to embed: 963


In [4]:
from tqdm import tqdm

texts = [r["chunk_text"] for r in rows]
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

In [5]:
conn = psycopg2.connect(DB_URL)
cur = conn.cursor()

for r, emb in tqdm(zip(rows, embeddings), total=len(rows)):
    cur.execute("""
        update knowledge_chunks
        set embedding = %s
        where id = %s
    """, (emb.tolist(), r["id"]))

conn.commit()
conn.close()

print("Embeddings stored successfully.")

100%|██████████| 963/963 [00:39<00:00, 24.69it/s]


Embeddings stored successfully.


In [29]:
def vector_search(query):
    conn = psycopg2.connect(DB_URL)
    cur = conn.cursor(cursor_factory=RealDictCursor)

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    cur.execute("""
        select *
        from match_knowledge_chunks(%s::vector(384), %s)
    """, (query_embedding.tolist(), 5))

    results = cur.fetchall()
    conn.close()

    for r in results:
        print("Similarity:", round(r["similarity"], 3))
        print(r["chunk_text"])
        print("-" * 50)

In [17]:
vector_search("How many hours of sleep are recommended for newborns and infants?")

Similarity: 0.622
## Getting enough sleep The daily recommended hours of sleep you need changes as you age. Recommended Daily Sleep by Age Group Newborn (0–3 months): 14–17 hours Infant (4–12 months): 12–16 hours (including naps) Toddler (1–2 years): 11–14 hours (including naps) Preschool (3–5 years): 10–13 hours (in
--------------------------------------------------
Similarity: 0.554
ref.org#bibliography-surfeit)). Human infants before the age of 3 mo do not display consolidated periods of non-REM and REM sleep. Indeed, REM sleep is quite difficult to discriminate in human infants at this age as it occurs often right after sleep onset, including also low-frequency EEG waves, wit
--------------------------------------------------
Similarity: 0.549
112,925. ### Sleep indicator Parents of children aged 3 to 5 years were asked, “During the past week, how many hours of sleep did this child get on an average day (count both nighttime sleep and naps)?” For ages 6 to 17 years, the question wa

In [21]:
vector_search("How many hours of sleep should adults get per day for optimal health?")

Similarity: 0.703
# About Sleep and Your Heart Health ## Key points * Getting good sleep isn’t just important for your energy levels—it’s critical for your heart health, too. * Sleep helps your body repair itself. Getting enough good sleep also helps you function normally during the day. ## How much sleep do I need? 
--------------------------------------------------
Similarity: 0.659
## Getting enough sleep The daily recommended hours of sleep you need changes as you age. Recommended Daily Sleep by Age Group Newborn (0–3 months): 14–17 hours Infant (4–12 months): 12–16 hours (including naps) Toddler (1–2 years): 11–14 hours (including naps) Preschool (3–5 years): 10–13 hours (in
--------------------------------------------------
Similarity: 0.628
## Overview Children and adolescents who do not get enough sleep have a higher risk for many health problems. Examples include obesity, type 2 diabetes, poor mental health, and injuries. These children are also more likely to have attention a

In [22]:
vector_search("What are the health risks associated with insufficient sleep among children and adolescents?")

Similarity: 0.776
### Summary What is already known about this topic? Insufficient sleep among children and adolescents is associated with an increased risk for obesity, diabetes, injuries, poor mental health, attention and behavior problems, and poor academic performance. Nationwide, approximately two thirds of U.S.
--------------------------------------------------
Similarity: 0.752
## Overview Children and adolescents who do not get enough sleep have a higher risk for many health problems. Examples include obesity, type 2 diabetes, poor mental health, and injuries. These children are also more likely to have attention and behavior problems, which can lead to poor academic perf
--------------------------------------------------
Similarity: 0.749
. ## Introduction Sleep is an essential function that predicts long-term health and well-being (1). Insufficient sleep has been associated with poor physical health (1–3), poor mental health, and problems with attention, behavior, learning, a

In [23]:
vector_search("Why is good sleep important for heart health?")

Similarity: 0.602
leep-and-heart-health.html#cdcreference_3) Some of these health problems raise the risk for heart disease, heart attack, and stroke. These health problems include: **High blood pressure.** During normal sleep, your blood pressure goes down. Having sleep problems means your blood pressure stays highe
--------------------------------------------------
Similarity: 0.568
# About Sleep and Your Heart Health ## Key points * Getting good sleep isn’t just important for your energy levels—it’s critical for your heart health, too. * Sleep helps your body repair itself. Getting enough good sleep also helps you function normally during the day. ## How much sleep do I need? 
--------------------------------------------------
Similarity: 0.551
 is valued and supported Parents strongly emphasized the value of healthy sleep practices and described their experiences supporting their child’s sleep. Parents explained that sleep was important for their child to function in everyday life 

In [24]:
vector_search("Is short sleep duration associated with obesity, diabetes, or cardiovascular disease in adults?")

Similarity: 0.674
9% for 8 hours, 17.2% for 9 hours, and 10.0% for ≥10 hours. The prevalence of short sleep duration in this combined sample was higher among female students (59.6%) than among male students (56.0%). The prevalence of short sleep duration also was highest among students in grade 6 (61.3%), lowest amon
--------------------------------------------------
Similarity: 0.646
 adults who usually slept less than 6 hours were more likely than adults who slept 7 to 8 hours to engage in certain health risk behaviors (i.e., cigarette smoking, having five or more drinks in a day, engaging in no leisure-time physical activity, and being obese). In many cases, adults who usually
--------------------------------------------------
Similarity: 0.646
leep-and-heart-health.html#cdcreference_3) Some of these health problems raise the risk for heart disease, heart attack, and stroke. These health problems include: **High blood pressure.** During normal sleep, your blood pressure goes down. H

In [25]:
vector_search("What is sleep hygiene, and how can it improve sleep quality?")

Similarity: 0.733
Paying attention to sleep hygiene is one of the most straightforward ways that you can set yourself up for better sleep. Strong sleep hygiene means having both a bedroom environment and daily routines that promote consistent, uninterrupted sleep. Every sleeper can tailor their sleep hygiene practice
--------------------------------------------------
Similarity: 0.686
 that reason, it’s worth testing out different adjustments to find out what helps your sleep the most. You don’t have to change everything at once; small steps can move you toward better sleep hygiene. It’s also important to know that improving sleep hygiene won’t always resolve sleeping problems. P
--------------------------------------------------
Similarity: 0.587
 child sleep on the individual, family, and community level. Attention to sleep as a risk for negative outcomes may be particularly important among children with MBDD. Conversely, addressing children’s mental health concerns and improving fac

In [26]:
vector_search("What is Shift Work Sleep Disorder (SWSD), and who is at risk?")

Similarity: 0.82
# **Shift Work Sleep Disorder (SWSD)** Shift work sleep disorder (SWSD) is a circadian rhythm sleep disorder that can affect people who work nontraditional hours. It causes issues with falling asleep, staying asleep and sleepiness at unwanted times. It’s treatable with lifestyle changes, light thera
--------------------------------------------------
Similarity: 0.786
 or other tests to make sure your symptoms aren’t due to another condition, such as [sleep apnea](https://my.clevelandclinic.org/health/diseases/8718-sleep-apnea), or medication side effects. ## **Management and Treatment** ### **How is shift work sleep disorder treated?** Although there’s no cure f
--------------------------------------------------
Similarity: 0.761
. For example, people with SWSD who work between 4 a.m. and 7 a.m. often have trouble falling asleep, while those who work in the evening tend to have issues staying asleep. * [Hypersomnia](https://my.clevelandclinic.org/health/diseases/21591-

In [27]:
vector_search("How can night shift work affect reproductive or overall health?")

Similarity: 0.735
## Why I should worry about my work schedule Shift work may increase your risk of reproductive health issues. Work schedule issues can include the following: * Working during the night * Shift work (working nights or rotating day, evening, and night shifts) * Irregular shifts and jet lag * Working l
--------------------------------------------------
Similarity: 0.576
 help treat [seasonal depression (seasonal affective disorder)](https://my.clevelandclinic.org/health/diseases/9293-seasonal-depression). Light therapy is especially helpful for night shift workers. #### **Melatonin** [Melatonin supplements](https://my.clevelandclinic.org/health/drugs/20908-melatoni
--------------------------------------------------
Similarity: 0.573
# **Shift Work Sleep Disorder (SWSD)** Shift work sleep disorder (SWSD) is a circadian rhythm sleep disorder that can affect people who work nontraditional hours. It causes issues with falling asleep, staying asleep and sleepiness at unwanted

In [30]:
vector_search("How does sleep duration and consistency affect academic performance in adolescents or college students?")

Similarity: 0.816
c.ncbi.nlm.nih.gov/articles/PMC8910766/#B28-ijerph-19-02933),[29](https://pmc.ncbi.nlm.nih.gov/articles/PMC8910766/#B29-ijerph-19-02933),[30](https://pmc.ncbi.nlm.nih.gov/articles/PMC8910766/#B30-ijerph-19-02933)\]. The negative influences of sleep problems on academic performance have also been reported in both children and college students, even though the mechanisms underlying the relationships remain to be explored. It has been widely observed that various sleep-related measures, such as sleep duration and sleep quality, are associated with academic performance \[[23](https://pmc.ncbi.nlm.nih.gov/articles/PMC8910766/#B23-ijerph-19-02933),[31](https://pmc.ncbi.nlm.nih.gov/articles/PMC8910766/#B31-ijerph-19-02933),[32](https://pmc.ncbi.nlm.nih.gov/articles/PMC8910766/#B32-ijerph-19-02933),[33](https://pmc.ncbi.nlm.nih.gov/articles/PMC8910766/#B33-ijerph-19-02933)\], indicating the importance of normal sleep patterns in school education. For college students, grade p

In [31]:
vector_search("What are the recommended physical activity guidelines for adults and children?")

Similarity: 0.722
## Overview The U.S. Department of Health and Human Services published the [*Physical Activity Guidelines for Americans*, 2nd edition](https://health.gov/sites/default/files/2019-09/Physical_Activity_Guidelines_2nd_edition.pdf) . The document recommends that children and adolescents aged 6–17 years do 60 minutes or more of moderate-to-vigorous physical activity daily. Regular physical activity in children and adolescents promotes health and fitness. Compared to their inactive peers, physically active youth have higher levels of fitness, lower body fat, and stronger bones and muscles. * Physical activity also has brain health benefits for school-aged children. These benefits include improved cognition (academic performance, memory) and reduced symptoms of depression. * Regular physical activity in childhood and adolescence can also be important for promoting lifelong health and well-being. Moreover, regular physical activity prevents risk factors for various health con

In [11]:
import re
def clean_text(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = s.replace("\u2013", "-")
    s = s.replace("\n", "\n")
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [10]:
import json

docs = json.load(open("articles.json", "r"))

In [16]:
for d in docs:
  print(f"Title = {d['title']} \n Snippet = {d['text'][:200]}")

Title = About Sleep 
 Snippet = ## Getting enough sleep

The daily recommended hours of sleep you need changes as you age.

Recommended Daily Sleep by Age Group 

Newborn (0–3 months): 14–17 hours

Infant (4–12 months): 12–16 hours 
Title = Sleep and Health 
 Snippet = ## Overview

Children and adolescents who do not get enough sleep have a higher risk for many health problems. Examples include obesity, type 2 diabetes, poor mental health, and injuries. These childr
Title = About Sleeping Sickness 
 Snippet = ## Overview

Sleeping sickness (i.e., human African trypanosomiasis or HAT) is caused by the parasite *Trypanosoma brucei*. A parasite is an organism (a living thing) that lives on or inside another o
Title = About Bed Bugs 
 Snippet = ## What to know

* Bed bugs are not known to spread diseases to people.
* Bites can cause itching, loss of sleep, and, rarely, allergic reactions.
* Prevent bed bugs by regularly looking for signs of 
Title = About Sleep and Your Heart Health 
 Snip